### Statistical Learning Models

#### Introduction

The goal of the modeling effort was to create a model of cost/watt in solar installations that accurately captures the price trend in order to make quarterly price predictions for the two years past the end of the data (2017, 2018).

The EDA phase suggested that time, size and location all had an impact on the cost/watt of a solar installation.  We used these features to model the cost. 

#### Features of the dataset

The dataset has some characteristics that impact modeling.  For example, the bulk of the installations are in the last three 
years (see figure xx).  Specifically, the raw dataset has about 711,000 installations in 2014, 2015 and 2016.  

Contrast this with 384,093 total installations for the previous 16 years, 1998 through 2013.  This means that grouping operations will have much larger groups in later years.  Group aggregates for earlier years may have erratic results.

The dataset is also quite noisy.  For example, Figure xxx is a histogram of prices paid in 2016 for installations in California between 4.5 and 5.5 kilowatts.  
 
	count	mean	std	min	25%	50%	75%	max
cost_per_watt	26107.0	4.410008	1.326826	1.0	3.744419	4.5	5.12	19.299145

There doesn’t seem to be much basis in the data for the large variance in price.  The lowest price among this group is $1.00/watt; the highest is $19.30.  50% of installations were priced between 3.75 and 5.12. That implies 50% are out of this central band.

Many modeling techniques rely on minimizing the squared difference between predicted and actual values.  As we see below, this accounts for relatively low metrics for models that seem to do a fairly good job of capturing the trend of the data.

#### The Modeling Strategy and Approach

The approach taken was to work from simpler to more detailed models (e.g., less features to more features; high bias to low bias models).  We used multiple kinds of techniques to be able compare results.

For every model, we made a graphic comparison of fitted and actual values to aid in assessment of the model.

For each model, if necessary, we analyzed hyper-parameter values for goodness of fit (bias/variance), choosing best hyper-parameters with best characteristics.

To tune hyper-parameters, the data was split into cross-validation set and test set.  Parameters were varied in a grid search and cross-validated.  Then, best cross-validated model was applied to the test set for performance measurement.

Where appropriate, we added regularization to evaluate its utility.

##### The Models

The following models were fitted and tuned for best performance. 
1.	Baseline OLS linear model: cost ~ time:  R^2: 4043
1.	OLS linear model: cost ~ time, size, state:  R^2:  0.4613
1.	OLS polynomial model: cost ~ time, size, state:  R^2: 0.5291
1.	Regularized versions of OLS polynomial models (Ridge/Lasso) showed no improvement in performance
1.	Random Forest Regressor with best parameters cost ~ time, size, state: R^2: 0.5993
1.  OLS linear model: median(cost) ~ time, weeks:  R^2:  0.715
1.	OLS polynomial model: median(cost) ~ time (weeks):  R^2: 0.8341
1.	OLS polynomial model: median(cost) ~ time(months), size:  R^2: 0.8512
1.  OLS polynomial model: median(cost) ~ time(months), state: R^2: 0.75

We began with an Ordinary Least Squares linear model with time as the single predictor to capture a baseline model and get a sense of the complexity in the data.  This simple model captures about 40% of the variance in the data.

Adding size and state to the linear model brought the R2 up to 0.46.

We then added a polynomial expansion to the OLS framework increasing the polynomial degree, stopping when the model was overfit, as indicated by a drop in R2 on the test set.  The best generalizable fit to the data was polynomial degree 14.  This much more flexible model had R2 of 0.53.

At this point, we experimented with L1 and L2 regularization (Lasso and Ridge Regression) to see if we could achieve better fit and generalization, varying both polynomial degree and the regularization parameter.  The best results were no better than the unregularized model.

Having gone as far as possible down this OLS-based path, we built a Random Forest model, a non-parametric model.  The hyper-parameters number of estimators and tree depth were explored with grid-search and cross-validation.  The model with best parameters had R2 of 0.60 on the test set, a decidely stronger value that OLS-based models. 

The Random Forest Regressor achieved a better R2 by capturing some characteristics of the noise.  This is visible in the graphic where outlying predictions are easily seen.

This resulted in a key insight.  We don't need a perfect model of solar pricing, we need a model that can be used to predict a Fair Model Value for solar installation over the next 12-24 months to answer the original business question.

This suggested another approach, modeling of the mean or median of the price within time windows.  Both the mean and the median provide noise rejection by aggregating many prices into a single number.  The random variations within the groups tend to cancel out leaving a less noisy data set.  The mean is however sensitive to outliers (there are many in the dataset).  The median rejects more noise since the amplitute of an extreme value is ignored.

We started by using an OLS linear model if the median cost at different time windows (day, week, month).  The best performing time frame for median was weeks with R2 of 0.715.

We continued with polynomial expansion of the median cost at different time windows (day, week, month).  The best performing time frame for median was weeks with R2 of 0.83 and polynomial degree

We then added size to the model by binning the size data and taking the median over combination of time interval and size.  This increased R2 to 0.85.  This did not improve performance, in fact reducing the best R2 to 0.75.

#### Predictions

While a strong R2 is an indication of a good fit to the dataset, for our purposes, we require a model that provide reasonable extrapolations for the 18-24 months past the end of the data since the shape of the extrapolation curve is fundamental to the business question at hand.  If the curve is flat or sloping upwards it is likely that there are no or little savings to be accrued by defering the purchase of a solar installation.  If the slope is moderately or steeply negative, waiting to purchase is likely to be rewarded with substantially lower cost.

As a result, we need not only an accurate fit (providing a good starting point for extrapolation, but also stable predictions, within market contraints.  The market constraints are two-fold:

1. Price is unlikely to increase substantially (dropping materials costs, vendor competition, continued government incentives, etc.)
2. Installation cost provides a positive floor for cost.

Below we compare each of the candidate predictions graphically.

The less flexible linear models seem to underestimate the rate of decline in cost, while the highly flexible OLS-based models seem generally to overestimate the rate (in two cases resulting in negative cost in 2018)

Only mod_06 captures the trend of the data while remaining positive throughout 2017 and 2018.

The Random Forest Regressor predicts a constant cost in 2017 and 2018.

The graph below depicts the average of all eight models compared to Model 6.  This is our prediction for the upper and lower bounds of the price in 6-8 quarters after the end of the data (2017-12-31).

#### Recommendations

The driving question for the product is << paste  here >>.

Having explored and modeled the data, we are in a position to make several recommendations.

1. Get several competitive quotations for the solar installation.

We can see that prices varied for installations in California in 2016 for similar size systems by a factor of 10.  A selection of vendors is likely to provide the best price.

2. Given the shape of the predicted cost curve, it is reasonable to expect that cost will drop by about $1.00/watt over 2017.  

It is also reasonable to expect at least this rate of decline in 2018, though it would be prudent to revisit this analysis incorporating more data as it becomes available.  In light of these predictions the customer can weigh the current cost of installation and the expected price decrease and make an informed decision.

#### Future work

The techniques used in this analysis are not the only modeling tools available.  In fact, there are a set of techniques specifically tailored to analysis and prediction of time series data.

A future project could perhaps usefully explore these tools in this context.  The current work would then provide a principled basis for comparison.